# Phase D: 路線価の地図可視化

3層構成で可視化する:
- **静止図** (matplotlib + contextily) — 論文・報告書用
- **Folium** — 手軽なインタラクティブマップ
- **pydeck** — 大規模データ対応の WebGL マップ

**合成データで動作確認済み。`DATA_PATH` を変えるだけで実データに適用可能。**

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import Normalize, TwoSlopeNorm
import contextily as ctx
import folium
import pydeck as pdk

try:
    import japanize_matplotlib
except ImportError:
    pass

from rex_io import load_and_prepare
from compute_change import add_distance_to_city

plt.rcParams["figure.dpi"] = 130

# ── パス設定 ──────────────────────────────────────────────────────────────
DATA_PATH = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")    
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

## 1. データ読み込み

In [ ]:
gdf = load_and_prepare(DATA_PATH)
# 変化率列を追加（なければ計算）
if "change_y1" not in gdf.columns:
    gdf["change_y1"] = (gdf["kakaku"] - gdf["pre1_kakak"]) / gdf["pre1_kakak"].replace(0, np.nan) * 100
    gdf["change_2y"] = (gdf["kakaku"] - gdf["pre2_kakak"]) / gdf["pre2_kakak"].replace(0, np.nan) * 100

print(f"件数: {len(gdf):,}  |  CRS: {gdf.crs}")

# 描画用に重心点を作成（矢線より点の方が地図描画が軽い）
gdf_pt = gdf.copy()
gdf_pt["geometry"] = gdf.geometry.centroid

## 2. 静止図（論文・報告書用）

### 2-1. 路線価の空間分布（コロプレス）

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# 路線価を対数スケールで色分け
gdf_plot = gdf_pt.to_crs(epsg=3857)
vmin, vmax = np.log10(gdf["kakaku"].quantile(0.05)), np.log10(gdf["kakaku"].quantile(0.95))
norm = Normalize(vmin=vmin, vmax=vmax)
colors = cm.YlOrRd(norm(np.log10(gdf_plot["kakaku"].clip(lower=1))))

gdf_plot.plot(
    ax=ax,
    color=colors,
    markersize=3,
    alpha=0.7,
)

# ベースマップ（OpenStreetMap）
try:
    ctx.add_basemap(ax, crs=gdf_plot.crs.to_string(), source=ctx.providers.CartoDB.Positron)
except Exception:
    pass  # オフライン環境ではスキップ

# カラーバー
sm = cm.ScalarMappable(cmap="YlOrRd", norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cb.set_label("log10(路線価) [千円/㎡]")
cb.set_ticks([1, 2, 3, 4])
cb.set_ticklabels(["10", "100", "1,000", "10,000"])

ax.set_title("路線価の空間分布（合成データ）", fontsize=13)
ax.set_axis_off()

plt.tight_layout()
plt.savefig(OUT_DIR / "map_price.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'map_price.png'}")

### 2-2. 年次変化率マップ（赤＝下落 / 青＝上昇）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

change_settings = [
    ("change_y1", "当年 vs 前年の変化率（%）"),
    ("change_2y",  "2年間の累計変化率（%）"),
]

for ax, (col, title) in zip(axes, change_settings):
    data = gdf_pt[col].dropna()
    gdf_sub = gdf_pt[gdf_pt[col].notna()].to_crs(epsg=3857)

    # ゼロを中央にした発散型カラーマップ
    abs_max = np.percentile(data.abs(), 95)
    norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
    colors = cm.RdBu(norm(gdf_sub[col]))

    gdf_sub.plot(ax=ax, color=colors, markersize=4, alpha=0.8)

    try:
        ctx.add_basemap(ax, crs=gdf_sub.crs.to_string(), source=ctx.providers.CartoDB.Positron)
    except Exception:
        pass

    sm = cm.ScalarMappable(cmap="RdBu", norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cb.set_label("変化率 (%)")
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()

plt.suptitle("路線価の年次変化率マップ（赤＝下落 / 青＝上昇）", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "map_change_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'map_change_rate.png'}")

### 2-3. 地区区分マップ

In [ ]:
CHIKU_COLORS = {
    "ビル街地区":           "#8B0000",
    "高度商業地区":          "#FF4500",
    "繁華街地区":           "#FF8C00",
    "普通商業・併用住宅地区": "#FFD700",
    "普通住宅地区":          "#4169E1",
    "中小工場地区":          "#32CD32",
    "大工場地区":           "#006400",
}

fig, ax = plt.subplots(figsize=(10, 8))
gdf_plot = gdf_pt.to_crs(epsg=3857)

for label, color in CHIKU_COLORS.items():
    subset = gdf_plot[gdf_plot["chikukbn_label"] == label]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, markersize=4, alpha=0.8, label=label)

try:
    ctx.add_basemap(ax, crs=gdf_plot.crs.to_string(), source=ctx.providers.CartoDB.Positron)
except Exception:
    pass

ax.legend(loc="lower right", fontsize=8, title="地区区分", title_fontsize=9)
ax.set_title("地区区分の空間分布", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT_DIR / "map_chikukbn.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {OUT_DIR / 'map_chikukbn.png'}")

## 3. Folium（インタラクティブHTML）

ポップアップでデータを確認できるインタラクティブマップ。  
`outputs/map_interactive.html` として保存する。

In [ ]:
def _price_to_color(price: float, vmin: float, vmax: float) -> str:
    norm = Normalize(vmin=vmin, vmax=vmax)
    rgba = cm.YlOrRd(norm(np.log10(max(price, 1))))
    return mcolors.to_hex(rgba)

def _change_to_color(change: float, abs_max: float) -> str:
    norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
    rgba = cm.RdBu(norm(np.clip(change, -abs_max, abs_max)))
    return mcolors.to_hex(rgba)

def _safe_float(val, default=0.0) -> float:
    """pd.NA / None / NaN を安全に float に変換する。"""
    try:
        f = float(val)
        return default if (f != f) else f  # NaN check
    except (TypeError, ValueError):
        return default

center = [35.68, 139.77]
m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")

log_vmin = np.log10(gdf["kakaku"].quantile(0.05))
log_vmax = np.log10(gdf["kakaku"].quantile(0.95))
abs_max  = _safe_float(gdf["change_y1"].abs().quantile(0.95), 5.0)

FOLIUM_MAX = 50_000
gdf_folium = gdf_pt.sample(min(FOLIUM_MAX, len(gdf_pt)), random_state=42) if len(gdf_pt) > FOLIUM_MAX else gdf_pt
print(f"Folium 描画件数: {len(gdf_folium):,} 件（全体: {len(gdf_pt):,} 件）")

price_layer  = folium.FeatureGroup(name="路線価（色=価格水準）", show=True)
change_layer = folium.FeatureGroup(name="変化率（赤=下落/青=上昇）", show=False)

for _, row in gdf_folium.iterrows():
    if row.geometry is None:
        continue
    lon, lat = row.geometry.x, row.geometry.y
    kakaku      = int(row["kakaku"])
    change_y1   = round(_safe_float(row.get("change_y1")), 2)
    change_2y   = round(_safe_float(row.get("change_2y")), 2)
    chiku_label = str(row.get("chikukbn_label", ""))

    popup_html = (
        f"<b>路線価</b>: {kakaku:,} 千円/㎡<br>"
        f"<b>地区区分</b>: {chiku_label}<br>"
        f"<b>変化率(前年比)</b>: {change_y1:+.2f}%<br>"
        f"<b>変化率(2年累計)</b>: {change_2y:+.2f}%"
    )

    folium.CircleMarker(
        location=[lat, lon], radius=4,
        color=_price_to_color(kakaku, log_vmin, log_vmax),
        fill=True, fill_opacity=0.75, weight=0.5,
        popup=folium.Popup(popup_html, max_width=200),
    ).add_to(price_layer)

    folium.CircleMarker(
        location=[lat, lon], radius=4,
        color=_change_to_color(change_y1, abs_max),
        fill=True, fill_opacity=0.75, weight=0.5,
        popup=folium.Popup(popup_html, max_width=200),
    ).add_to(change_layer)

price_layer.add_to(m)
change_layer.add_to(m)
folium.LayerControl().add_to(m)

out_html = OUT_DIR / "map_interactive.html"
m.save(str(out_html))
print(f"保存: {out_html}")
m

## 4. pydeck（大規模データ対応 WebGL マップ）

230万件の実データも描画できる WebGL レンダリング。  
`outputs/map_pydeck_price.html` / `map_pydeck_change.html` として保存。

In [ ]:
def _make_pydeck_df(gdf_: gpd.GeoDataFrame) -> pd.DataFrame:
    """pydeck 用 DataFrame を作成（lon/lat/color列を追加）。"""
    df = gdf_.copy()
    df["lon"] = df.geometry.x
    df["lat"] = df.geometry.y
    return df[["lon", "lat", "kakaku", "change_y1", "change_2y",
               "chikukbn_label", "swari_ratio"]].dropna(subset=["lon", "lat"])

def _kakaku_to_rgb(df: pd.DataFrame) -> pd.DataFrame:
    """路線価を [R, G, B] に変換（YlOrRd 風）。"""
    log_p = np.log10(df["kakaku"].clip(lower=1))
    norm  = (log_p - log_p.quantile(0.05)) / (log_p.quantile(0.95) - log_p.quantile(0.05))
    norm  = norm.clip(0, 1)
    # YlOrRd: 黄(255,255,0) → 赤(139,0,0)
    r = (255 * (1 - norm * 0.45)).astype(int)
    g = (255 * (1 - norm)).astype(int).clip(0, 255)
    b = np.zeros(len(df), dtype=int)
    df = df.copy()
    df["color"] = list(zip(r, g, b, [180] * len(df)))
    return df

def _change_to_rgb(df: pd.DataFrame, col: str = "change_y1") -> pd.DataFrame:
    """変化率を [R, G, B]（赤=下落/青=上昇）に変換。"""
    v = df[col].fillna(0)
    abs_max = v.abs().quantile(0.95)
    norm = (v / abs_max).clip(-1, 1)
    # 赤(214,96,77) → 白 → 青(33,102,172)
    r = np.where(norm < 0, 214, (214 * (1 - norm)).astype(int).clip(0, 214))
    g = (96 + (255 - 96) * (1 - norm.abs())).astype(int).clip(0, 255)
    b = np.where(norm > 0, 172, (172 * (1 + norm)).astype(int).clip(0, 172))
    df = df.copy()
    df["color"] = list(zip(r.astype(int), g.astype(int), b.astype(int), [200] * len(df)))
    return df

# データ準備
pdk_df_price  = _kakaku_to_rgb(_make_pydeck_df(gdf_pt))
pdk_df_change = _change_to_rgb(_make_pydeck_df(gdf_pt), col="change_y1")

center_lon = float(gdf_pt.geometry.x.mean())
center_lat = float(gdf_pt.geometry.y.mean())
view = pdk.ViewState(latitude=center_lat, longitude=center_lon, zoom=6, pitch=0)

# ── 路線価マップ ──────────────────────────────────────────────────────────
layer_price = pdk.Layer(
    "ScatterplotLayer",
    data=pdk_df_price,
    get_position="[lon, lat]",
    get_fill_color="color",
    get_radius=300,
    pickable=True,
    opacity=0.8,
)
deck_price = pdk.Deck(
    layers=[layer_price],
    initial_view_state=view,
    tooltip={"text": "路線価: {kakaku}千円/㎡\n地区: {chikukbn_label}"},
    map_style="light",
)
deck_price.to_html(str(OUT_DIR / "map_pydeck_price.html"))
print(f"保存: {OUT_DIR / 'map_pydeck_price.html'}")

# ── 変化率マップ ──────────────────────────────────────────────────────────
layer_change = pdk.Layer(
    "ScatterplotLayer",
    data=pdk_df_change,
    get_position="[lon, lat]",
    get_fill_color="color",
    get_radius=300,
    pickable=True,
    opacity=0.8,
)
deck_change = pdk.Deck(
    layers=[layer_change],
    initial_view_state=view,
    tooltip={"text": "変化率(前年比): {change_y1}%\n変化率(2年): {change_2y}%\n地区: {chikukbn_label}"},
    map_style="light",
)
deck_change.to_html(str(OUT_DIR / "map_pydeck_change.html"))
print(f"保存: {OUT_DIR / 'map_pydeck_change.html'}")

## 5. 出力ファイルまとめ

In [ ]:
outputs = list(OUT_DIR.glob("map_*"))
print("=== outputs/ に保存されたマップ ===")
for p in sorted(outputs):
    size_kb = p.stat().st_size // 1024
    print(f"  {p.name:40s}  {size_kb:>6,} KB")